# simpleFoam residuals analysis

This notebook parses a `simpleFoam` (or `simpleFoamExtracted`) log file and produces:

1. **Residual history** — initial residuals per field on a semilogy scale, one line per field.
2. **Execution time per iteration** — wall-clock cost of each SIMPLE step.
3. **Summary table** — final residual values and convergence status.

**Dependencies:** Python standard library + `numpy` + `matplotlib` only (no pandas).

Set `LOG_FILE` below to the path of your log file before running all cells.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
LOG_FILE = "../log.txt"   # path to your simpleFoam log file

# Fields to look for.  The parser is not limited to this list —
# any field found in the log will be plotted — but this order
# controls the legend sort.
FIELD_ORDER = ["Ux", "Uy", "Uz", "p", "k", "epsilon", "omega", "nuTilda"]

## 1 · Parse the log file

The parser recognises two log line formats produced by OpenFOAM linear solvers:

```
smoothSolver:  Solving for Ux, Initial residual = 0.000122, Final residual = 1.22e-05, No Iterations 5
GAMG:  Solving for p, Initial residual = 0.00131, Final residual = 0.000116, No Iterations 4
```

and the execution-time line:

```
ExecutionTime = 6.06 s  ClockTime = 6 s
```

Each `ExecutionTime` line marks the end of one SIMPLE iteration.

In [ ]:
import re
import os
from collections import defaultdict

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# ── Regex patterns ────────────────────────────────────────────────────────────
_RE_SOLVER = re.compile(
    r"(?:smoothSolver|GAMG|PCG|PBiCGStab|diagonal):\s+"
    r"Solving for (\w+),\s+"
    r"Initial residual = ([\d.eE+\-]+),\s+"
    r"Final residual = ([\d.eE+\-]+),\s+"
    r"No Iterations (\d+)"
)
_RE_TIME = re.compile(
    r"ExecutionTime\s*=\s*([\d.]+)\s*s\s+ClockTime\s*=\s*([\d.]+)\s*s"
)


def parse_log(path):
    """Parse a simpleFoam log file.

    Returns
    -------
    residuals : dict[field -> list[float]]
        Initial residual per SIMPLE iteration for each field.
    final_res : dict[field -> list[float]]
        Final (inner) residual per iteration.
    exec_times : list[float]
        Cumulative ExecutionTime in seconds at the end of each iteration.
    n_iters : dict[field -> list[int]]
        Number of inner iterations per SIMPLE step.
    """
    residuals  = defaultdict(list)
    final_res  = defaultdict(list)
    n_iters    = defaultdict(list)
    exec_times = []

    # Buffer current-iteration solver lines until we see ExecutionTime
    _buf = defaultdict(list)   # field -> [(init, final, nit), ...]

    with open(path, "r", errors="replace") as fh:
        for line in fh:
            m = _RE_SOLVER.search(line)
            if m:
                field = m.group(1)
                _buf[field].append((
                    float(m.group(2)),   # initial residual
                    float(m.group(3)),   # final residual
                    int(m.group(4)),     # iterations
                ))
                continue

            m = _RE_TIME.search(line)
            if m:
                exec_times.append(float(m.group(1)))
                # Flush buffer: take the LAST solve of each field in this step
                # (some fields are solved multiple times per iteration, e.g.
                # non-orthogonal correctors for p)
                for field, entries in _buf.items():
                    init = entries[-1][0]
                    fin  = entries[-1][1]
                    nit  = entries[-1][2]
                    residuals[field].append(init)
                    final_res[field].append(fin)
                    n_iters[field].append(nit)
                _buf.clear()

    return dict(residuals), dict(final_res), exec_times, dict(n_iters)


# ── Load ──────────────────────────────────────────────────────────────────────
if not os.path.exists(LOG_FILE):
    raise FileNotFoundError(
        f"Log file not found: {LOG_FILE!r}\n"
        "Set LOG_FILE to the path of your simpleFoam log."
    )

residuals, final_res, exec_times, n_iters = parse_log(LOG_FILE)

n_steps = len(exec_times)
print(f"Parsed {n_steps} SIMPLE iterations from {LOG_FILE!r}")
print(f"Fields found: {sorted(residuals.keys())}")

## 2 · Plot 1: Initial residuals vs iteration

Each line shows how the **initial residual** of one field evolves over the SIMPLE
iterations.  This is the standard convergence diagnostic:

- A monotonically decreasing curve means the solver is converging.
- A plateau means relaxation is too tight or the mesh is limiting convergence.
- A rising or oscillating curve (especially for p or k) indicates potential divergence.

The semilogy scale lets you see several decades of residual drop simultaneously.

In [ ]:
# Colour palette — enough for 8 fields
_PALETTE = [
    "#4e79a7", "#f28e2b", "#59a14f", "#e15759",
    "#76b7b2", "#edc948", "#b07aa1", "#ff9da7",
]

# Sort fields: prefer FIELD_ORDER, append any extras alphabetically
_extras = sorted(k for k in residuals if k not in FIELD_ORDER)
_sorted_fields = [f for f in FIELD_ORDER if f in residuals] + _extras

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#16213e")

for idx, field in enumerate(_sorted_fields):
    colour = _PALETTE[idx % len(_PALETTE)]
    y = residuals[field]
    x = np.arange(1, len(y) + 1)
    ax.semilogy(x, y, label=field, color=colour, linewidth=1.4)

ax.set_xlabel("SIMPLE iteration", color="#ccc")
ax.set_ylabel("Initial residual", color="#ccc")
ax.set_title("Residual history", color="#7ecfff", fontsize=13)
ax.tick_params(colors="#ccc")
ax.spines[:].set_color("#334")
ax.grid(True, which="both", color="#2a2a4a", linewidth=0.6, linestyle="--")
ax.legend(framealpha=0.15, labelcolor="white", facecolor="#0f1a2e",
          edgecolor="#334", fontsize=9)

plt.tight_layout()
plt.savefig("residuals.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Saved residuals.png")

## 3 · Plot 2: Execution time per iteration

The log reports **cumulative** `ExecutionTime`.  Taking the difference between
consecutive values gives the cost of each individual SIMPLE step.

This plot is useful for:
- Detecting outlier iterations caused by sudden mesh-cell activation (adaptive
  refinement) or load imbalance in parallel.
- Comparing solver configurations (GAMG vs PCG, different smoother sweeps).
- Estimating total run time from the early iterations.

In [ ]:
cum = np.array(exec_times)
# Delta time per step (first step: just use its cumulative value directly)
dt = np.diff(cum, prepend=0.0)
dt[dt < 0] = 0.0   # guard against log restarts / clock resets

iterations = np.arange(1, len(cum) + 1)

fig2, (ax_dt, ax_cum) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
fig2.patch.set_facecolor("#1a1a2e")

for ax in (ax_dt, ax_cum):
    ax.set_facecolor("#16213e")
    ax.tick_params(colors="#ccc")
    ax.spines[:].set_color("#334")
    ax.grid(True, color="#2a2a4a", linewidth=0.6, linestyle="--")

ax_dt.bar(iterations, dt, color="#4e79a7", alpha=0.8, width=0.8)
ax_dt.set_ylabel("Step time (s)", color="#ccc")
ax_dt.set_title("Execution time per SIMPLE iteration", color="#7ecfff", fontsize=13)
_mean_dt = dt.mean()
ax_dt.axhline(_mean_dt, color="#ffd", linewidth=1, linestyle="--",
              label=f"mean = {_mean_dt:.3f} s")
ax_dt.legend(framealpha=0.15, labelcolor="white", facecolor="#0f1a2e",
             edgecolor="#334", fontsize=9)

ax_cum.plot(iterations, cum, color="#f28e2b", linewidth=1.5)
ax_cum.set_xlabel("SIMPLE iteration", color="#ccc")
ax_cum.set_ylabel("Cumulative time (s)", color="#ccc")

plt.tight_layout()
plt.savefig("exec_time.png", dpi=150, bbox_inches="tight",
            facecolor=fig2.get_facecolor())
plt.show()
print("Saved exec_time.png")

## 4 · Summary table

Final residuals and convergence status for each field.

A field is considered **converged** when its final initial residual is at or
below `1e-4` (the typical `residualControl` target).  Adjust the threshold
below if your case uses tighter or looser criteria.

In [ ]:
CONVERGENCE_THRESHOLD = 1e-4

# ── Build table data ──────────────────────────────────────────────────────────
rows = []
for field in _sorted_fields:
    r = residuals[field]
    f = final_res[field]
    ni = n_iters[field]
    converged = r[-1] <= CONVERGENCE_THRESHOLD if r else False
    rows.append((
        field,
        f"{r[0]:.4e}" if r else "—",
        f"{r[-1]:.4e}" if r else "—",
        f"{f[-1]:.4e}" if f else "—",
        f"{ni[-1]}" if ni else "—",
        "YES" if converged else "no",
    ))

# ── ASCII table ───────────────────────────────────────────────────────────────
header = ("Field", "Init res (iter 1)", "Init res (final)",
          "Final res (final)", "Inner iters", "Converged?")
col_w = [max(len(header[i]), max((len(r[i]) for r in rows), default=0))
         for i in range(len(header))]

sep = "+" + "+".join("-" * (w + 2) for w in col_w) + "+"

def _fmt_row(cells):
    return "|" + "|".join(f" {c:<{col_w[i]}} " for i, c in enumerate(cells)) + "|"

print(sep)
print(_fmt_row(header))
print(sep)
for row in rows:
    print(_fmt_row(row))
print(sep)

print(f"\nTotal iterations: {n_steps}")
print(f"Total execution time: {cum[-1]:.1f} s  ({cum[-1]/60:.1f} min)")
all_converged = all(r[-1] <= CONVERGENCE_THRESHOLD for _, r in residuals.items() if r)
if all_converged:
    print(f"\nConverged in {n_steps} iterations (all fields below {CONVERGENCE_THRESHOLD:.0e}).")
else:
    not_converged = [f for f in _sorted_fields
                     if residuals.get(f) and residuals[f][-1] > CONVERGENCE_THRESHOLD]
    print(f"\nNot yet converged: {not_converged}")
    print("Consider running more iterations or checking relaxation factors.")

## 5 · Plot 3: Final vs initial residual ratio

The ratio **final / initial residual** tells you how effectively the linear
solver is reducing the error *within each SIMPLE step* (inner convergence).

- A ratio close to `relTol` (typically 0.1) means the solver is exiting via
  the relative tolerance — normal and efficient.
- A ratio much larger than `relTol` means the solver hit `maxIter` before
  converging — consider increasing `maxIter` or switching solver.
- A ratio much smaller than `relTol` means the absolute `tolerance` was hit —
  the inner system is over-solved; you can raise `tolerance` to save time.

In [ ]:
fig3, ax3 = plt.subplots(figsize=(10, 4))
fig3.patch.set_facecolor("#1a1a2e")
ax3.set_facecolor("#16213e")
ax3.tick_params(colors="#ccc")
ax3.spines[:].set_color("#334")
ax3.grid(True, which="both", color="#2a2a4a", linewidth=0.6, linestyle="--")

for idx, field in enumerate(_sorted_fields):
    colour = _PALETTE[idx % len(_PALETTE)]
    ri = np.array(residuals[field])
    rf = np.array(final_res[field])
    # Avoid division by zero
    ratio = np.where(ri > 0, rf / ri, np.nan)
    x = np.arange(1, len(ratio) + 1)
    ax3.semilogy(x, ratio, label=field, color=colour, linewidth=1.2, alpha=0.85)

# Reference lines for typical relTol values
ax3.axhline(0.1,  color="#ffd", linewidth=0.8, linestyle=":",  alpha=0.6, label="relTol=0.1")
ax3.axhline(0.05, color="#bdf", linewidth=0.8, linestyle="--", alpha=0.5, label="relTol=0.05")

ax3.set_xlabel("SIMPLE iteration", color="#ccc")
ax3.set_ylabel("Final / Initial residual", color="#ccc")
ax3.set_title("Inner solver efficiency (final/initial residual ratio)",
              color="#7ecfff", fontsize=12)
ax3.legend(framealpha=0.15, labelcolor="white", facecolor="#0f1a2e",
           edgecolor="#334", fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig("solver_efficiency.png", dpi=150, bbox_inches="tight",
            facecolor=fig3.get_facecolor())
plt.show()
print("Saved solver_efficiency.png")

## 6 · Export residual data to CSV

For use in external tools (spreadsheets, other scripts) without any pandas
dependency — just the standard library `csv` module.

In [ ]:
import csv

OUT_CSV = "residuals.csv"

with open(OUT_CSV, "w", newline="") as fh:
    writer = csv.writer(fh)
    # Header
    header_fields = [f"{field}_init" for field in _sorted_fields] + \
                    [f"{field}_final" for field in _sorted_fields] + \
                    ["exec_time_s"]
    writer.writerow(["iteration"] + header_fields)

    for step in range(n_steps):
        row = [step + 1]
        for field in _sorted_fields:
            arr = residuals.get(field, [])
            row.append(arr[step] if step < len(arr) else "")
        for field in _sorted_fields:
            arr = final_res.get(field, [])
            row.append(arr[step] if step < len(arr) else "")
        row.append(exec_times[step] if step < len(exec_times) else "")
        writer.writerow(row)

print(f"Wrote {n_steps} rows to {OUT_CSV!r}")